In [0]:
import json
import uuid
from datetime import datetime, timezone

from pyspark.sql import functions as F


dbutils.widgets.text("environment", "dev")
dbutils.widgets.text("qa_run_id", "")
dbutils.widgets.text("triggered_by", "")
dbutils.widgets.text("job_id", "")
dbutils.widgets.text("job_run_id", "")


def _param(name):
    value = dbutils.widgets.get(name).strip()
    # Unresolved dynamic references (e.g. when run interactively) count as empty.
    return "" if value.startswith("{{") else value


environment = dbutils.widgets.get("environment").lower()
catalog = f"retail_{environment}"
started_at = datetime.now(timezone.utc)

job_id = _param("job_id")
job_run_id = _param("job_run_id")

# Custom, human-readable id for this QA run, e.g. QA-PROD-20261002-143015-a1b2c3.
# The app passes its own id so it can track the run before it finishes.
qa_run_id = _param("qa_run_id") or (
    f"QA-{environment.upper()}-{started_at:%Y%m%d-%H%M%S}-{(job_run_id or uuid.uuid4().hex)[-6:]}"
)
triggered_by = _param("triggered_by") or ("job" if job_run_id else "manual")

print(f"QA run id: {qa_run_id} (triggered by {triggered_by}, job run {job_run_id or 'n/a'})")

results = []


def record(test_name, layer, status, actual=None, expected=None, details=None):
    results.append(
        {
            "test_name": test_name,
            "layer": layer.upper(),
            "status": status,
            "actual": actual,
            "expected": expected,
            "details": details,
        }
    )


def table_exists(full_name):
    try:
        return spark.catalog.tableExists(full_name)
    except Exception:
        return False


def check_table(name, layer):
    full_name = f"{catalog}.{layer}.{name}"
    ok = table_exists(full_name)

    record(
        f"{layer}.{name} exists",
        layer,
        "PASS" if ok else "FAIL",
        actual=ok,
        expected=True,
    )

    return full_name if ok else None


def check_columns(full_name, required, layer):
    actual = set(spark.table(full_name).columns)
    missing = sorted(set(required) - actual)
    ok = not missing

    record(
        f"{full_name} required columns",
        layer,
        "PASS" if ok else "FAIL",
        actual=sorted(actual),
        expected=required,
        details=f"Missing: {missing}" if missing else None,
    )


def check_non_empty(full_name, layer):
    count = spark.table(full_name).count()
    ok = count > 0

    record(
        f"{full_name} non-empty",
        layer,
        "PASS" if ok else "FAIL",
        actual=count,
        expected="> 0",
    )


def check_unique(full_name, keys, layer):
    df = spark.table(full_name)

    duplicate_groups = (
        df.groupBy(*keys)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    record(
        f"{full_name} business-key uniqueness",
        layer,
        "PASS" if duplicate_groups == 0 else "FAIL",
        actual=duplicate_groups,
        expected=0,
        details=f"Keys: {keys}",
    )


# -------------------------
# Bronze technical QA
# -------------------------

# These are the columns that every Bronze table must contain.
# _rescued_data and _corrupt_record are intentionally NOT mandatory
# because they are technical columns that may not exist when there
# are no rescued/corrupt records.
bronze_required_specs = {
    "customers": [
        "customer_id",
        "email",
        "_source_file",
        "_ingestion_timestamp",
        "_load_date",
        "_batch_id",
        "_environment",
    ],
    "products": [
        "product_id",
        "_source_file",
        "_ingestion_timestamp",
        "_load_date",
        "_batch_id",
        "_environment",
    ],
    "orders": [
        "order_id",
        "_source_file",
        "_ingestion_timestamp",
        "_load_date",
        "_batch_id",
        "_environment",
    ],
    "clickstream": [
        "event_id",
        "_source_file",
        "_ingestion_timestamp",
        "_load_date",
        "_batch_id",
        "_environment",
    ],
    "returns": [
        "return_id",
        "_source_file",
        "_ingestion_timestamp",
        "_load_date",
        "_batch_id",
        "_environment",
    ],
    "promotions": [
        "promotion_id",
        "_source_file",
        "_ingestion_timestamp",
        "_load_date",
        "_batch_id",
        "_environment",
    ],
}


for dataset, columns in bronze_required_specs.items():
    full_name = check_table(dataset, "bronze")

    if full_name:
        check_columns(full_name, columns, "BRONZE")
        check_non_empty(full_name, "BRONZE")

        df = spark.table(full_name)

        rescued_column_present = "_rescued_data" in df.columns
        corrupt_column_present = "_corrupt_record" in df.columns

        rescued = (
            df.filter(F.col("_rescued_data").isNotNull()).count()
            if rescued_column_present
            else 0
        )

        corrupt = (
            df.filter(F.col("_corrupt_record").isNotNull()).count()
            if corrupt_column_present
            else 0
        )

        record(
            f"{full_name} technical ingestion quality",
            "BRONZE",
            "PASS" if rescued == 0 and corrupt == 0 else "WARN",
            actual={
                "rescued": rescued,
                "corrupt": corrupt,
                "rescued_column_present": rescued_column_present,
                "corrupt_column_present": corrupt_column_present,
            },
            expected={
                "rescued": 0,
                "corrupt": 0,
            },
            details=(
                "Optional technical columns are absent."
                if not rescued_column_present or not corrupt_column_present
                else None
            ),
        )


# Promotions is a snapshot.
# Current fixture is expected to contain 200 source rows.
promotions = f"{catalog}.bronze.promotions"

if table_exists(promotions):
    promotion_count = spark.table(promotions).count()

    record(
        "Bronze promotions snapshot row count",
        "BRONZE",
        "PASS" if promotion_count == 200 else "WARN",
        actual=promotion_count,
        expected=200,
        details=(
            "Warn rather than fail so a legitimate source snapshot "
            "change does not block QA."
        ),
    )


# -------------------------
# Silver business QA
# -------------------------

silver_required = {
    "customers": ["customer_id", "email"],
    "products": ["product_id"],
    "orders": ["order_id"],
    "clickstream": ["event_id"],
    "returns": [
        "return_id",
        "order_id",
        "customer_id",
        "product_id",
    ],
    "promotions": ["promotion_id"],
}


for dataset, columns in silver_required.items():
    full_name = check_table(dataset, "silver")

    if full_name:
        check_columns(full_name, columns, "SILVER")
        check_non_empty(full_name, "SILVER")


for dataset, keys in {
    "customers": ["customer_id"],
    "products": ["product_id"],
    "orders": ["order_id"],
    "clickstream": ["event_id"],
    "returns": ["return_id"],
    "promotions": ["promotion_id"],
}.items():
    full_name = f"{catalog}.silver.{dataset}"

    if table_exists(full_name):
        check_unique(full_name, keys, "SILVER")


dq_table = f"{catalog}.monitoring.silver_dq_metrics"

if table_exists(dq_table):
    from pyspark.sql import Window

    dq = spark.table(dq_table)

    # Latest batch per dataset (one test per dataset, not the latest N rows overall).
    latest = (
        dq.withColumn(
            "_rn",
            F.row_number().over(Window.partitionBy("dataset").orderBy(F.col("recorded_at").desc())),
        )
        .filter(F.col("_rn") == 1)
        .orderBy("dataset")
        .collect()
    )

    for row in latest:
        status = row["dq_status"]

        record(
            f"Silver DQ latest status: {row['dataset']}",
            "SILVER",
            "PASS" if status in {"PASS", "ALERT"} else "FAIL",
            actual=status,
            expected="PASS or ALERT",
            details=(
                f"failure_rate={row['dq_failure_rate_percent']:.2f}%, "
                f"threshold={row['dq_failure_threshold_percent']:.2f}%"
            ),
        )


# -------------------------
# Gold QA
# -------------------------

gold_unique_specs = {
    "dim_customer": [
        "customer_id",
        "effective_start_date",
    ],
    "dim_product": ["product_id"],
    "fact_orders": ["order_id"],
    "fact_returns": ["return_id"],
}


for dataset, keys in gold_unique_specs.items():
    full_name = check_table(dataset, "gold")

    if full_name:
        check_non_empty(full_name, "GOLD")
        check_unique(full_name, keys, "GOLD")


dim_customer = f"{catalog}.gold.dim_customer"

if table_exists(dim_customer):
    df = spark.table(dim_customer)

    current_count = (
        df.filter(F.col("is_current") == True)
        .count()
    )

    invalid_current = (
        df.filter(
            (F.col("is_current") == True)
            & (
                F.col("effective_end_date")
                != F.lit("9999-12-31").cast("date")
            )
        )
        .count()
    )

    record(
        "Gold dim_customer SCD2 current-row integrity",
        "GOLD",
        "PASS" if invalid_current == 0 else "FAIL",
        actual={
            "current_rows": current_count,
            "invalid_current_rows": invalid_current,
        },
        expected=(
            "Every current row has "
            "effective_end_date=9999-12-31"
        ),
    )


# -------------------------
# Gold referential integrity: every return should reference an existing order
# -------------------------

fact_returns = f"{catalog}.gold.fact_returns"
fact_orders_table = f"{catalog}.gold.fact_orders"

if table_exists(fact_returns) and table_exists(fact_orders_table):
    orphans = spark.sql(f"""
        SELECT count(*) AS orphan_returns, coalesce(round(sum(r.refund_amount), 2), 0) AS orphan_refunds,
               (SELECT count(*) FROM {fact_returns}) AS total_returns
        FROM {fact_returns} r
        LEFT ANTI JOIN (SELECT DISTINCT order_id FROM {fact_orders_table}) o ON r.order_id = o.order_id
    """).first()

    record(
        "Gold fact_returns reference existing orders",
        "GOLD",
        "PASS" if orphans["orphan_returns"] == 0 else "WARN",
        actual={
            "orphan_returns": orphans["orphan_returns"],
            "total_returns": orphans["total_returns"],
            "orphan_refund_amount": float(orphans["orphan_refunds"]),
        },
        expected={"orphan_returns": 0},
        details=(
            "Returns whose order_id is not in fact_orders are excluded from revenue summaries."
            if orphans["orphan_returns"] else None
        ),
    )


# -------------------------
# Gold summary reconciliation
# -------------------------

fact_orders = f"{catalog}.gold.fact_orders"
customer_summary = f"{catalog}.gold.customer_sales_summary"


if table_exists(fact_orders) and table_exists(customer_summary):
    fact_df = spark.table(fact_orders)
    summary_df = spark.table(customer_summary)

    candidate_fact = [
        c
        for c in [
            "net_sales_amount",
            "net_amount",
            "net_sales",
            "revenue",
            "sales_amount",
        ]
        if c in fact_df.columns
    ]

    candidate_summary = [
        c
        for c in [
            "net_sales_amount",
            "net_sales",
            "net_amount",
            "revenue",
            "sales_amount",
            "total_sales",
        ]
        if c in summary_df.columns
    ]

    if candidate_fact and candidate_summary:
        fact_column = candidate_fact[0]
        summary_column = candidate_summary[0]

        fact_total = (
            fact_df
            .agg(F.sum(fact_column))
            .first()[0]
            or 0.0
        )

        summary_total = (
            summary_df
            .agg(F.sum(summary_column))
            .first()[0]
            or 0.0
        )

        delta = abs(
            float(fact_total) - float(summary_total)
        )

        record(
            "Gold customer summary reconciles to fact_orders",
            "GOLD",
            "PASS" if delta < 0.01 else "FAIL",
            actual={
                "fact_column": fact_column,
                "summary_column": summary_column,
                "fact_total": float(fact_total),
                "summary_total": float(summary_total),
                "delta": delta,
            },
            expected="delta < 0.01",
        )

    else:
        record(
            "Gold customer summary reconciliation",
            "GOLD",
            "WARN",
            actual={
                "fact_columns": fact_df.columns,
                "summary_columns": summary_df.columns,
            },
            expected="Recognized sales measure columns",
            details=(
                "Skipped because the existing summary schema "
                "does not expose a recognized measure column."
            ),
        )


# -------------------------
# Persist results + exit
# -------------------------

failed = sum(
    1 for r in results
    if r["status"] == "FAIL"
)

warnings = sum(
    1 for r in results
    if r["status"] == "WARN"
)

passed = sum(
    1 for r in results
    if r["status"] == "PASS"
)


finished_at = datetime.now(timezone.utc)
total = len(results)
run_status = "FAIL" if failed else ("WARN" if warnings else "PASS")
pass_rate = round(passed / total * 100, 2) if total else 0.0

qa_summary = {
    "qa_run_id": qa_run_id,
    "environment": environment,
    "catalog": catalog,
    "status": run_status,
    "run_timestamp_utc": finished_at.isoformat(),
    "started_at_utc": started_at.isoformat(),
    "duration_seconds": round((finished_at - started_at).total_seconds(), 1),
    "triggered_by": triggered_by,
    "job_run_id": job_run_id,
    "summary": {
        "passed": passed,
        "failed": failed,
        "warnings": warnings,
        "total": total,
        "pass_rate_percent": pass_rate,
    },
    "tests": results,
}


spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.monitoring")

# -------------------------
# Per-test results (one row per test, tagged with the run id)
# -------------------------

target = f"{catalog}.monitoring.qa_test_results"

# qa_run_id goes last so existing columns keep their positions.
result_rows = [
    (
        environment,
        r["layer"],
        r["test_name"],
        r["status"],
        json.dumps(r["actual"], default=str),
        json.dumps(r["expected"], default=str),
        r.get("details"),
        finished_at,
        qa_run_id,
    )
    for r in results
]

result_schema = """
environment string,
layer string,
test_name string,
status string,
actual string,
expected string,
details string,
recorded_at timestamp,
qa_run_id string
"""

(
    spark.createDataFrame(result_rows, result_schema)
    .write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(target)
)

# One-time backfill: runs written before qa_run_id existed get a stable legacy id
# derived from the minute they were recorded in. No-op once every row has an id.
spark.sql(f"""
    UPDATE {target}
    SET qa_run_id = concat('QA-', upper(environment), '-',
                           date_format(date_trunc('MINUTE', recorded_at), 'yyyyMMdd-HHmm'), '-legacy')
    WHERE qa_run_id IS NULL
""")

# -------------------------
# Run summary (one row per QA run)
# -------------------------

summary_table = f"{catalog}.monitoring.qa_run_summary"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {summary_table} (
        qa_run_id STRING COMMENT 'Custom QA run id, e.g. QA-PROD-20261002-143015-a1b2c3',
        environment STRING,
        catalog STRING,
        status STRING COMMENT 'PASS, WARN (warnings only) or FAIL (at least one failed test)',
        total_tests INT,
        passed INT,
        warnings INT,
        failed INT,
        pass_rate_percent DOUBLE,
        started_at TIMESTAMP,
        finished_at TIMESTAMP,
        duration_seconds DOUBLE,
        triggered_by STRING,
        job_id STRING,
        job_run_id STRING
    ) USING DELTA
    COMMENT 'One row per QA automation run; test-level rows live in qa_test_results'
""")

spark.createDataFrame(
    [(
        qa_run_id, environment, catalog, run_status, total, passed, warnings, failed, pass_rate,
        started_at, finished_at, round((finished_at - started_at).total_seconds(), 1),
        triggered_by, job_id or None, job_run_id or None,
    )],
    """
    qa_run_id string, environment string, catalog string, status string, total_tests int,
    passed int, warnings int, failed int, pass_rate_percent double, started_at timestamp,
    finished_at timestamp, duration_seconds double, triggered_by string, job_id string,
    job_run_id string
    """,
).createOrReplaceTempView("qa_run_summary_source")

spark.sql(f"""
    MERGE INTO {summary_table} AS t
    USING qa_run_summary_source AS s
    ON t.qa_run_id = s.qa_run_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

# Summaries for legacy runs (rebuilt from their test rows; idempotent).
spark.sql(f"""
    MERGE INTO {summary_table} AS t
    USING (
        SELECT
            qa_run_id,
            first(environment) AS environment,
            '{catalog}' AS catalog,
            CASE WHEN sum(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END) > 0 THEN 'FAIL'
                 WHEN sum(CASE WHEN status = 'WARN' THEN 1 ELSE 0 END) > 0 THEN 'WARN'
                 ELSE 'PASS' END AS status,
            CAST(count(*) AS INT) AS total_tests,
            CAST(sum(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END) AS INT) AS passed,
            CAST(sum(CASE WHEN status = 'WARN' THEN 1 ELSE 0 END) AS INT) AS warnings,
            CAST(sum(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END) AS INT) AS failed,
            round(sum(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END) * 100.0 / count(*), 2) AS pass_rate_percent,
            min(recorded_at) AS started_at,
            max(recorded_at) AS finished_at,
            CAST(NULL AS DOUBLE) AS duration_seconds,
            'legacy' AS triggered_by,
            CAST(NULL AS STRING) AS job_id,
            CAST(NULL AS STRING) AS job_run_id
        FROM {target}
        WHERE qa_run_id LIKE '%-legacy'
        GROUP BY qa_run_id
    ) AS s
    ON t.qa_run_id = s.qa_run_id
    WHEN NOT MATCHED THEN INSERT *
""")

print(f"Persisted {total} results for {qa_run_id} → {target} and {summary_table}")


print(
    json.dumps(
        qa_summary,
        indent=2,
        default=str,
    )
)

dbutils.notebook.exit(
    json.dumps(
        qa_summary,
        default=str,
    )
)